In [2]:
# 라이브러리 설치
# !pip install konlpy

In [3]:
from konlpy.tag import Okt
okt = Okt()
print(okt.morphs('나는 학교에 간다'))

['나', '는', '학교', '에', '간다']


In [4]:
import jpype
# jpype.startJVM(r'C:\Program Files\Java\jdk-17\bin\server\jvm.dll')
print('JVM : ', jpype.isJVMStarted())

JVM :  True


# 데이터의 분할
- KFold의 분할 방식
    - kFold
        - 무작위로 데이터를 폴드화
- StratifiedKFold
    - 계층화를 유지하며 폴드화

In [5]:
import pandas as pd
from sklearn.model_selection import KFold,StratifiedGroupKFold, StratifiedKFold

data = {
    'document' : ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H'],
    'label' : [1,1,0,0,1,0,0,1],
    'id' : ['a', 'a', 'b', 'b', 'c', 'c', 'd', 'd']
}
df = pd.DataFrame(data)
df

,document,label,id
0,A,1,a
1,B,1,a
2,C,0,b
3,D,0,b
4,E,1,c
5,F,0,c
6,G,0,d
7,H,1,d


In [6]:
# 일반적인 KFold

X = df['document']
Y = df['label']
groups = df['id']

k_folds = KFold(n_splits=2, shuffle=True, random_state=42)
s_folds = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
sg_fold = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=42)

In [7]:
for x_idx, y_idx in (k_folds.split(X, Y)):
    print(df.loc[x_idx])
    print(df.loc[y_idx])
    break

  document  label id
2        C      0  b
3        D      0  b
4        E      1  c
6        G      0  d
  document  label id
0        A      1  a
1        B      1  a
5        F      0  c
7        H      1  d


In [8]:
# 계층화 KFold
for x_idx, y_idx in s_folds.split(X, Y):
    print(df.loc[x_idx])
    print(df.loc[y_idx])
    break

  document  label id
1        B      1  a
3        D      0  b
6        G      0  d
7        H      1  d
  document  label id
0        A      1  a
2        C      0  b
4        E      1  c
5        F      0  c


In [9]:
df = pd.read_csv("../data/ratings_train.txt", sep='\t')

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [11]:
df.loc[~df.isna().any(axis=1)]

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
...,...,...,...
149995,6222902,인간이 문제지.. 소는 뭔죄인가..,0
149996,8549745,평점이 너무 낮아서...,1
149997,9311800,이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?,0
149998,2376369,청춘 영화의 최고봉.방황과 우울했던 날들의 자화상,1


In [12]:
df.dropna(inplace=True)

In [13]:
df['document'].value_counts()

document
굿                                                181
good                                              92
최고                                                85
쓰레기                                               79
별로                                                66
                                                ... 
굿바이 레닌 표절인것은 이해하는데 왜 뒤로 갈수록 재미없어지냐                 1
이건 정말 깨알 캐스팅과 질퍽하지않은 산뜻한 내용구성이 잘 버무러진 깨알일드!!♥      1
약탈자를 위한 변명, 이라. 저놈들은 착한놈들 절대 아닌걸요.                 1
나름 심오한 뜻도 있는 듯. 그냥 학생이 선생과 놀아나는 영화는 절대 아님          1
흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나                  1
Name: count, Length: 146182, dtype: int64

In [14]:
before = len(df)

df = df.drop_duplicates(['document']).reset_index(drop=True)
after = len(df)

In [15]:
print("중복 데이터를 제거한 뒤 행의 개수 : ", before - after)

중복 데이터를 제거한 뒤 행의 개수 :  3813


In [16]:
print(len(df['id'].unique()))

146182


In [17]:
df['label'].value_counts()

label
0    73342
1    72840
Name: count, dtype: int64

In [18]:
from sklearn.model_selection import train_test_split
X = df['document'].values
Y = df['label'].values

# X_temp, X_test, Y_temp, Y_test = train_test_split(
#     X, Y, 
#     test_size=0.2,
#     random_state=42
# )
# X_test , X_val, Y_test, Y_val = train_test_split(
#     X_test, Y_test,
#     test_size=0.5,
#     random_state=42
# )

# validation 데이터셋을 11% 정도로 나눠준다 (label 데이터의 비율에 맞게)
X_temp, X_test, Y_temp, Y_test = train_test_split(
    X, Y, test_size=0.1, random_state=42, stratify=Y
)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_temp, Y_temp, test_size=0.11, random_state=42, stratify=Y_temp
)


In [19]:
# 데이터셋의 분할 정도를 확인
print(len(X_train)/ len(X) * 100)
print(len(X_val) / len(X) * 100)
print(len(X_test) / len(X) * 100)

80.09946505041661
9.89998768658248
10.000547263000916


In [20]:
print(pd.Series(Y_train).value_counts())

0    58746
1    58345
Name: count, dtype: int64


In [21]:
print(pd.Series(Y_val).value_counts())

0    7261
1    7211
Name: count, dtype: int64


In [22]:
# 계층 폴드화 -> 학습 데이터를 데이터 분할 / 학습하여 일반적인 성능을 나타내는
# 폴드화, 하이퍼 파라미터 탐색과 같이 사용
folds = []

# enumeratie() -> 리스트에서 위치와 값으로 데이터를 나눠서 돌려준다
# s_folds.split(X_train, Y_train)
for fold, (tr_idx, va_idx) in enumerate(
    s_folds.split(X_train, Y_train)
):
    folds.append(
        {
            'fold' : fold,
            'tr_idx' : tr_idx,
            'va_idx' : va_idx
        }
    )

folds

[{'fold': 0,
  'tr_idx': array([     0,      2,      3, ..., 117083, 117086, 117090],
        shape=(58545,)),
  'va_idx': array([     1,      7,      8, ..., 117087, 117088, 117089],
        shape=(58546,))},
 {'fold': 1,
  'tr_idx': array([     1,      7,      8, ..., 117087, 117088, 117089],
        shape=(58546,)),
  'va_idx': array([     0,      2,      3, ..., 117083, 117086, 117090],
        shape=(58545,))}]

In [23]:
# fold에서 첫번째 데이터에서 tr_idx 가지고 Y_train의 0, 1의 비율을 확인
test_idx = folds[0]['tr_idx']
pd.Series(Y_train[test_idx]).value_counts()

0    29373
1    29172
Name: count, dtype: int64

# 단어의 토큰화
- 문장을 단어로 잘라준다.
    - 공백을 기준으로 문자를 자른다.
        - 영문에서는 사용 가능, 한글에서는 의미가 소실되는 경우가 발생
    - 형태소를 사용하여 문자를 나눠준다
        - 국어 사전을 로드하여 단어별로 나눠준다.

In [24]:
# 공백을 기반으로 데이터를 나눈다.
test = '나는 학교에 간다'

In [25]:
tokens = test.split()
print(tokens)

['나는', '학교에', '간다']


In [ ]:
from konlpy.tag import Okt
okt = Okt()

print(okt.morphs(test))

print(okt.pos(test))

['나', '는', '학교', '에', '간다']
[('나', 'Noun'), ('는', 'Josa'), ('학교', 'Noun'), ('에', 'Josa'), ('간다', 'Noun')]


In [27]:
test_pos = okt.pos(test)

In [30]:
# 반복문을 이용하여 각원소들을 대입하여 실행
for t in test_pos:
    # print(t)
    # t -> tuple -> 두번째 값이 'Josa'가 아니라면
    if t[1] != 'Josa':
        print(t[0])

나
학교
간다


In [42]:
# Okt 로드한 데이터를 이용하여 Okt 형태소 분석
for idx, t in enumerate(X_train):
    if idx == 5:
        break
    _pos = okt.morphs(t)
    print(_pos)

['이런', '감동', '...', '삶', '의', '희망이', '된다']
['초등학교', '때', '이', '거', '200', '번', '받음', '..', '정말', '짱']
['아이엠', '옴티머', '스프', '라임']
['나', '만', '재밋', '게', '봤나']
['최고', '의', '영화', '평점', '1', '점주', '는', '초딩', '들', '은', '대체', '뭐', '냐', '?', '요새', '한국', '영화', '들', '보다', '훨', '낫다', '.', '영화', '보고', '10년', '넘게', '기억', '에', '남았던', '명작', '이다', '.']


In [49]:
# !pip install Korpora

In [50]:
# from Korpora import Korpora
# data = Korpora.load('nsmc')

In [52]:
# 형태소를 이용한 토큰화
# !pip install sentencepiece

In [55]:
# sentencepiece 모듈을 이용하여 형태소 분석
# 모델 학습
# train txt, test txt 파일을 모드 로드하여 학습에 대입
df_tr = pd.read_csv("../data/ratings_train.txt", sep='\t').dropna()
df_te = pd.read_csv("../data/ratings_test.txt", sep='\t').dropna()


In [57]:
total_df = pd.concat( [df_tr['document'], df_te['document']], axis=0, ignore_index=True)

In [58]:
total_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 199992 entries, 0 to 199991
Series name: document
Non-Null Count   Dtype 
--------------   ----- 
199992 non-null  object
dtypes: object(1)
memory usage: 1.5+ MB


In [59]:
# 모델에 학습시키기 전에 파일로 미리 저장
total_df.to_csv('test.txt', index=False, header=False)

In [60]:
import sentencepiece as spm

spm.SentencePieceTrainer.Train(
    input = 'test.txt', # 학습에서 사용할 텍스트 파일
    model_prefix = 'ko_unigram', # unigram -> 한글 적합한 형태 (한단어씩 잘라서 표현)
    vocab_size = 8000, # 단어 사전의 크기(모델의 크기) -> 8000, 16000, 32000
    model_type = 'unigram', # 토큰의 생성 방식, unigram -> BERT, KoGPT 등에서 사용되는 언어 방식
                            # bpe -> GPT-2 사용하는 방식 (한글에서 적합X)
                            # char -> 문자 단위(정보가 너무 짧게 쪼개져 있는 형태)
                            # word -> 단어 기준
    character_coverage = 0.9995, # 학습에 포함할 문자 종류의 비율
                                 # 1.0인 경우 모든 문자의 종류를 사용
                                 # 0.9995 -> 한글, 영문, 숫자 포함
    input_sentence_size = 100000, # 학습 문장을 샘플링
                                  # 모든 데이터를 사용하는게 가장 좋은 방법
                                  # 시간상의 문제로 일부만 샘플링하여 사용하는 방법
    shuffle_input_sentence = True # 샘플링시 문장의 순서를 섞어서 사용하는 방법
                                  # 모델이 특정 순서에 편향되는것을 방지
)

In [61]:
# 생성된 모델을 이용하여 형태소 분석
sp = spm.SentencePieceProcessor()
# 생성된 모델을 로드
sp.load('ko_unigram.model')
text = '나는 학교에 간다'
print(sp.encode(text, out_type = str))

['▁나는', '▁학교', '에', '▁간다']


In [63]:
# ▁ 특수 기호는 키보드 입력이 불가
char = '\u2581'
print(char)

▁


In [64]:
for idx, t in enumerate(X_train):
    if idx == 5:
        break
    print(sp.encode(t, out_type=str))

['▁이런', '▁감동', '...', '삶', '의', '▁희망', '이', '▁된다']
['▁초', '등', '학교', '▁때', '▁이거', '▁200', '번', '▁받', '음', '..', '▁정말', '▁짱']
['▁아이', '엠', '옴', '티', '머', '스', '프', '라', '임']
['▁나만', '▁재밋게', '▁봤', '나']
['▁최고의', '▁영화', '▁평점', '1', '점주는', '▁초딩들', '은', '▁대체', '▁뭐냐', '?', '▁요새', '▁한국영화', '들', '보다', '▁훨', '▁낫다', '.', '▁영화보고', '▁10', '년', '넘', '게', '▁기억에', '▁남', '았던', '▁명작이다', '.']


In [65]:
from konlpy.tag import Komoran

In [66]:
komoran = Komoran()

In [68]:
print(komoran.morphs(text)) # 형태소 나열
print(komoran.pos(text)) # (형태소, 동사) 튜플 나열
print(komoran.nouns(text)) # 명사만 출력

['나', '는', '학교', '에', '간다']
[('나', 'NP'), ('는', 'JX'), ('학교', 'NNG'), ('에', 'JKB'), ('간다', 'NNP')]
['학교', '간다']


### komoran 동사를 일반적으로 사용하는 것들
- 감성/의도 분석/리뷰 (가장 일반적)
    - NNG(일반병사), NNP(고유명사), VV(동사), VA(형용사), MAG(일반부사),
    SL(외국어)
- 명사 기반의 분류 (문서에 대한 분류 작업)
    - NNG(일반명사), NNP(고유명사), NR(수사), NP(대명사)
- 의미가 있는 단어를 최대한 포함하고 싶은 경우
    - NNG(일반명사), NNP(고유명사), VV(동사), VA(형용사), MAG(일반부사), MAJ(접속부사),
    IC(감탄사), SL(외국어)
    

In [71]:
allow_pos = ['NNG', 'NNP', 'VV', 'VA']

# 선택한 형태소들을 추출하기 위한 함수를 정의
def komoran_tokenize(text):
    result = []
    for morph, pos in komoran.pos(text):
        if pos in allow_pos:
            result.append(morph)
    return result
    
for idx, t in enumerate(X_train):
    if idx == 5:
        break
    print(komoran_tokenize(t))

['감동', '삶', '희망', '되']
['초등학교', '때', '받']
[]
['보']
['최고', '영화', '평점', '주', '초딩', '대체', '요새', '한국', '영화', '낫', '영화', '넘', '기억', '남', '명작']


# 벡터화
- 토큰화 작업에서 단어들을 추출했다면 해당 단어들을 숫자형으로 변환
    - 숫자형태로 변환하는 이유는? -> 컴퓨터가 숫자로만 연산이 가능하기 때문에
- 숫자형태로 변환한 데이터를 학습 데이터로 이용, 정답은 lebel 데이터로 규칙을 생성해가는 과정

In [72]:
# one-hot encoding -> 하나의 리뷰에서 특정 단어가 포함되어 있는가?
df = pd.read_csv("../data/ratings_train.txt", sep='\t').dropna()
df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [73]:
# 전체의 텍스트를 이용해서 학습을 통한 단어를 습득한 뒤
# 해당하는 단어들이 리뷰에 포함되어 있는가?

from sklearn.feature_extraction.text import CountVectorizer

In [76]:
# 객체 생성
# 존재 여부만 파악 객체 생성
vectorizer = CountVectorizer(binary=True)

# 모델 학습 변환(data 대입) -> 데이터는 document에서 10개의 데이터
X = vectorizer.fit_transform(df['document'].head(10))
X

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 71 stored elements and shape (10, 70)>

In [75]:
# 학습한 단어들이 무엇인가 출력 (단어 사전)
vocab = vectorizer.get_feature_names_out()
print(vocab)

['1학년생인' '3세부터' '8살용영화' 'ㅋㅋㅋ' '가볍지' '가족도없다' '감금만반복반복' '걸음마' '교도소' '그것보단'
 '긴장감을' '길들여져' '길용우' '납치' '낫겟다' '낮은건데' '너무' '너무나도' '너무재밓었다그래서보는것을추천한다'
 '늙어보이기만' '더빙' '던스트가' '돋보였던' '몇안되는' '목소리' '반개도' '발로해도' '별반개도' '볼만한데'
 '사이몬페그의' '살려내지못했다' '솔직히' '스파이더맨에서' '아까움' '아깝다' '않구나' '액션이' '없는데도' '없다'
 '연기가' '연기못하는사람만모엿네' '연기생활이몇년인지' '영화' '오버연기조차' '왜케' '욕나온다' '원작의' '이드라마는'
 '이뻐보였다' '이야기구먼' '이응경' '익살스런' '있나' '있는' '재미' '재미는' '정말' '제대로' '조정' '진짜'
 '짜증나네요' '초등학교' '초딩영화줄' '커스틴' '평점' '평점이' '포스터보고' '했던' '헐리우드식' '화려함에만']


In [77]:
print(X.toarray())

[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1
  0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0
  0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 1 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 1 1 0 0 0 0 0 0 1 0 0 1 0 0 0
  0 0 0 1 0 0 1 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0]
 [1 1 1 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 

In [81]:
# Okt + CounterVectorizer 같이 사용 -> 토큰화 + 벡터화

okt = Okt()

def okt_tokenize(text):
    select_pos = ['Noun', 'Verb', 'Adjective']
    # (단어, 형태)를 출력하는 pos() 함수 사용
    # result = okt.morphs(text)
    # return result
    result = [
        word for word, pos in okt.pos(text) if pos in select_pos
    ]
    result2 = []
    for word, pos in okt.pos(text):
        if pos in select_pos:
            result2.append(word) #result랑 같음
    return result
    

# CounterVectorizer 생성
vectorizer_okt = CountVectorizer(
    tokenizer = okt_tokenize,
    lowercase= False,
    binary=True,
)

x_okt = vectorizer_okt.fit_transform(df['document'].head(5))

c:\Users\johnh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [82]:
print(vectorizer_okt.get_feature_names_out())

['가볍지' '교도소' '구먼' '늙어' '다그' '더빙' '던스트' '돋보였던' '래서' '목소리' '몬페' '무재' '밓었'
 '보고' '보는것을' '보였다' '보이기만' '솔직히' '스파이더맨' '않구나' '없다' '연기' '영화' '오버' '의' '이뻐'
 '이야기' '익살스런' '재미' '조정' '줄' '진짜' '짜증나네요' '초딩' '추천' '커스틴' '평점' '포스터' '했던'
 '흠']


In [83]:
print(x_okt.toarray())

[[0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0
  0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 1 1 1 0 0 0 0 0 0 1 0 0 1 0 0
  0 1 0 1]
 [0 0 0 0 1 0 0 0 1 0 0 1 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0
  0 0 0 0]
 [0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 1 0 1 1 0 0 0 0 0 0
  1 0 0 0]
 [0 0 0 1 0 0 1 1 0 0 1 0 0 0 0 1 1 0 1 0 0 1 1 0 1 1 0 1 0 0 0 0 0 0 0 1
  0 0 1 0]]


In [84]:
pd.DataFrame(
    x_okt.toarray(),
    columns=vectorizer_okt.get_feature_names_out()
)

,가볍지,교도소,구먼,늙어,다그,더빙,던스트,돋보였던,래서,목소리,...,줄,진짜,짜증나네요,초딩,추천,커스틴,평점,포스터,했던,흠
0,0,0,0,0,0,1,0,0,0,1,...,0,1,1,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,1,0,0,1,0,0,0,1,0,1
2,0,0,0,0,1,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
3,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,0,0,0,1,0,0,1,1,0,0,...,0,0,0,0,0,1,0,0,1,0
